# 02 — TFT Model Evaluation

This notebook evaluates the trained Temporal Fusion Transformer against naive baselines
from Notebook 01, analyses prediction errors by temporal slice, extracts feature importance
and attention weights, and compares discovered bias patterns to the hardcoded `BIAS_TABLE`.

**Prerequisites**: Run `training/train.py` so that a checkpoint exists in `training/models/`.

| Section | Purpose |
|:--------|:--------|
| 1 | Load model & data |
| 2 | Overall metrics vs naive baselines |
| 3 | Error analysis by hour, day-of-week, season |
| 4 | Feature importance (encoder variable selection) |
| 5 | Attention weight analysis |
| 6 | Discovered bias vs hardcoded BIAS_TABLE |
| 7 | Worst prediction deep-dive |

In [ ]:
import os, sys, pickle, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', font_scale=0.9)

PROJECT_ROOT = str(Path.cwd().parent)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src', 'api'))
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'training'))

from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.metrics import MAE
from train import create_datasets  # type: ignore[import-not-found]

MODEL_DIR = Path(PROJECT_ROOT) / 'training' / 'models'
TRAINING_DIR = Path(PROJECT_ROOT) / 'training'
print(f'Project root : {PROJECT_ROOT}')
print(f'Model dir    : {MODEL_DIR}')
print(f'Torch device : {torch.device("cuda" if torch.cuda.is_available() else "cpu")}')

## 1 — Load Model & Data

In [ ]:
# Find best checkpoint
ckpts = sorted(MODEL_DIR.glob('*.ckpt'))
if not ckpts:
    raise FileNotFoundError(
        f'No checkpoints found in {MODEL_DIR}.\n'
        'Run: python training/train.py   first.'
    )

# Pick the checkpoint with the lowest val_MAE in the filename
import re
def _mae_from_name(p):
    m = re.search(r'val_MAE=(\d+)', p.stem) or re.search(r'-(\d+)$', p.stem)
    return int(m.group(1)) if m else 999999

best_ckpt = min(ckpts, key=_mae_from_name)
print(f'Best checkpoint: {best_ckpt.name}  (val_MAE ≈ {_mae_from_name(best_ckpt)})')

model = TemporalFusionTransformer.load_from_checkpoint(best_ckpt)
model.eval()
_ = model.to('cpu')

print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Load training data and recreate splits
data_files = sorted(TRAINING_DIR.glob('tft_training_data*.csv'), reverse=True)
if not data_files:
    raise FileNotFoundError('No training CSV found. Run complete_data.py first.')

# Prefer -main, else latest versioned
main_file = [f for f in data_files if 'main' in f.name]
data_path = main_file[0] if main_file else data_files[0]
print(f'Data: {data_path.name}')

df = pd.read_csv(data_path)
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)
df['time_idx'] = range(len(df))
df['group'] = 'Germany'
df['season'] = df['season'].astype(str)

# Same split ratios as train.py
TRAIN_RATIO, VAL_RATIO = 0.80, 0.10
train_end = int(len(df) * TRAIN_RATIO)
val_end = int(len(df) * (TRAIN_RATIO + VAL_RATIO))

train_df = df[:train_end].copy()
val_df = df[train_end:val_end].copy()
test_df = df[val_end:].copy()

print(f'Total: {len(df):,}  |  Train: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test: {len(test_df):,}')
print(f'Test range: {test_df["timestamp"].min()} → {test_df["timestamp"].max()}')

In [ ]:
# Rebuild datasets for prediction
dataset_pkl = MODEL_DIR / 'training_dataset.pkl'
if dataset_pkl.exists():
    with open(dataset_pkl, 'rb') as f:
        training_dataset = pickle.load(f)
    print('Loaded training_dataset.pkl')
else:
    # Reconstruct from train.py settings
    from train import create_datasets  # type: ignore[import-not-found]
    training_dataset, _ = create_datasets(df)
    print('Reconstructed training dataset')

# Create test dataset from training template
test_dataset = TimeSeriesDataSet.from_dataset(training_dataset, test_df, predict=True)
test_loader = test_dataset.to_dataloader(train=False, batch_size=512, num_workers=0)

val_dataset = TimeSeriesDataSet.from_dataset(training_dataset, val_df, predict=True)
val_loader = val_dataset.to_dataloader(train=False, batch_size=512, num_workers=0)

print(f'Test batches : {len(test_loader)}')

## 2 — Overall Metrics vs Naive Baselines

In [ ]:
# Generate predictions on test set
raw_preds = model.predict(test_loader, mode='raw', return_x=True, trainer_kwargs={'accelerator': 'cpu'})

# Extract point predictions and actuals
preds = raw_preds.output['prediction'].squeeze(-1)   # (n_samples, pred_len)
# Get the index mapping back to the dataframe
decoder_target = raw_preds.output['decoder_target']   # (n_samples, pred_len)

print(f'Prediction tensor shape: {preds.shape}')
print(f'Target tensor shape    : {decoder_target.shape}')

In [ ]:
# Flatten to per-timestep for metrics
pred_flat = preds.detach().cpu().numpy().flatten()
actual_flat = decoder_target.detach().cpu().numpy().flatten()

# Core metrics
def compute_metrics(actual, predicted, label=''):
    err = predicted - actual
    mae = np.mean(np.abs(err))
    rmse = np.sqrt(np.mean(err**2))
    mape = np.mean(np.abs(err / np.where(actual == 0, 1, actual))) * 100
    bias = np.mean(err)
    return {'Model': label, 'MAE (MW)': mae, 'RMSE (MW)': rmse, 'MAPE (%)': mape, 'Bias (MW)': bias, 'n': len(actual)}

tft_metrics = compute_metrics(actual_flat, pred_flat, 'TFT')

# Naive baselines on the TEST split for fair comparison
test_actual = test_df['target_demand'].values
test_ts = test_df['timestamp']

# Persistence-24h baseline (from full df so we can look back)
full_demand = df.set_index('timestamp')['target_demand']
test_persist_24 = test_ts.map(lambda t: full_demand.get(t - pd.Timedelta(hours=24), np.nan)).values
test_persist_168 = test_ts.map(lambda t: full_demand.get(t - pd.Timedelta(hours=168), np.nan)).values

mask_24 = ~np.isnan(test_persist_24)
mask_168 = ~np.isnan(test_persist_168)

baselines = [
    compute_metrics(test_actual[mask_24], test_persist_24[mask_24], 'Persistence-24h'),
    compute_metrics(test_actual[mask_168], test_persist_168[mask_168], 'Persistence-168h'),
]

results = pd.DataFrame([tft_metrics] + baselines).round(1)
results = results.set_index('Model')
print('Test Set Performance')
results

In [ ]:
# Metric comparison bar chart
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, metric in zip(axes, ['MAE (MW)', 'RMSE (MW)', 'MAPE (%)']):
    colors = ['#2ca02c', '#1f77b4', '#ff7f0e'][:len(results)]
    results[metric].plot.barh(ax=ax, color=colors)  # type: ignore[arg-type]
    ax.set_title(metric)
    ax.invert_yaxis()
fig.suptitle('TFT vs Naive Baselines — Test Set', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 3 — Error Analysis by Hour, Day-of-Week, Season

In [ ]:
# Build a per-step results dataframe
# Each prediction window is 24 steps; we need to align timestamps
PRED_LEN = preds.shape[1]
ENCODER_LEN = 48

# Get the decoder time indices from raw_preds
decoder_time_idx = raw_preds.x['decoder_time_idx'].cpu().numpy()  # (n_samples, pred_len)

rows = []
for i in range(preds.shape[0]):
    for j in range(PRED_LEN):
        tidx = int(decoder_time_idx[i, j])
        rows.append({
            'time_idx': tidx,
            'pred': float(preds[i, j]),
            'actual': float(decoder_target[i, j]),
            'horizon': j + 1,
        })

step_df = pd.DataFrame(rows)
# Average over duplicate time_idx (same timestep predicted by multiple windows)
step_agg = step_df.groupby('time_idx').agg(
    pred=('pred', 'mean'),
    actual=('actual', 'first'),
    horizon=('horizon', 'mean'),
).reset_index()

# Merge with timestamps
idx_to_ts = df.set_index('time_idx')['timestamp']
step_agg['timestamp'] = step_agg['time_idx'].map(idx_to_ts)
step_agg['error'] = step_agg['pred'] - step_agg['actual']
step_agg['abs_error'] = step_agg['error'].abs()
step_agg['hour'] = step_agg['timestamp'].dt.hour
step_agg['dow'] = step_agg['timestamp'].dt.dayofweek
step_agg['month'] = step_agg['timestamp'].dt.month
step_agg['season'] = step_agg['month'].map({
    12: 'Winter', 1: 'Winter', 2: 'Winter',
    3: 'Spring', 4: 'Spring', 5: 'Spring',
    6: 'Summer', 7: 'Summer', 8: 'Summer',
    9: 'Autumn', 10: 'Autumn', 11: 'Autumn',
})

print(f'Unique timesteps evaluated: {len(step_agg):,}')
step_agg.head()

In [ ]:
# MAE by hour of day
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# By hour
hour_mae = step_agg.groupby('hour')['abs_error'].mean()
axes[0].bar(hour_mae.index, hour_mae.values, color='steelblue')
axes[0].set_xlabel('Hour')
axes[0].set_ylabel('MAE (MW)')
axes[0].set_title('MAE by Hour of Day')

# By dow
dow_labels = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
dow_mae = step_agg.groupby('dow')['abs_error'].mean()
colors = ['coral' if d >= 5 else 'steelblue' for d in dow_mae.index]
axes[1].bar(dow_labels, dow_mae.values, color=colors)
axes[1].set_ylabel('MAE (MW)')
axes[1].set_title('MAE by Day of Week')

# By season
season_order = ['Winter', 'Spring', 'Summer', 'Autumn']
season_mae = step_agg.groupby('season')['abs_error'].mean().reindex(season_order)
axes[2].bar(season_order, season_mae.values, color=['#4c72b0', '#55a868', '#c44e52', '#dd8452'])
axes[2].set_ylabel('MAE (MW)')
axes[2].set_title('MAE by Season')

fig.suptitle('TFT Error Breakdown', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Bias heatmap (hour × dow) for TFT
tft_bias_pivot = step_agg.pivot_table(index='hour', columns='dow', values='error', aggfunc='mean')
tft_bias_pivot.columns = dow_labels

fig, ax = plt.subplots(figsize=(8, 10))
vmax = max(abs(tft_bias_pivot.values.min()), abs(tft_bias_pivot.values.max()))
sns.heatmap(tft_bias_pivot, annot=True, fmt='.0f', cmap='RdBu_r',
            center=0, vmin=-vmax, vmax=vmax, ax=ax)
ax.set_title('TFT Mean Bias by Hour × Day-of-Week (MW)', fontsize=12)
ax.set_ylabel('Hour')
plt.tight_layout()
plt.show()

In [ ]:
# MAE by forecast horizon (1-24h)
horizon_mae = step_df.groupby('horizon')['pred'].apply(
    lambda x: (x - step_df.loc[x.index, 'actual']).abs().mean()
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(horizon_mae.index, horizon_mae.to_numpy(), 'o-', color='steelblue', markersize=5)
ax.set_xlabel('Forecast Horizon (hours)')
ax.set_ylabel('MAE (MW)')
ax.set_title('MAE by Forecast Horizon (1–24h ahead)')
ax.set_xticks(range(1, 25))
ax.axhline(hour_mae.mean(), ls='--', color='grey', alpha=0.5, label='Overall MAE')
ax.legend()
plt.tight_layout()
plt.show()

## 4 — Feature Importance (Encoder Variable Selection Weights)

In [ ]:
# TFT provides encoder variable selection weights
interpretation = model.interpret_output(raw_preds.output, reduction='mean')

# Encoder attention (known reals + categoricals)
enc_importance = (
    pd.Series(
        interpretation['encoder_variables'].detach().cpu().numpy(),
        index=model.encoder_variables
    )
    .sort_values(ascending=False)
)

# Plot top 25
top_n = min(25, len(enc_importance))
fig, ax = plt.subplots(figsize=(8, max(6, top_n * 0.3)))
enc_importance.head(top_n).plot.barh(ax=ax, color='steelblue')
ax.invert_yaxis()
ax.set_xlabel('Variable Selection Weight')
ax.set_title(f'Top {top_n} Encoder Variable Importance')
plt.tight_layout()
plt.show()

print(f'\nTop 10 features:')
for feat, weight in enc_importance.head(10).items():
    print(f'  {weight:.4f}  {feat}')

In [ ]:
# Decoder variable importance
dec_importance = (
    pd.Series(
        interpretation['decoder_variables'].detach().cpu().numpy(),
        index=model.decoder_variables
    )
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(8, max(4, len(dec_importance) * 0.3)))
dec_importance.plot.barh(ax=ax, color='coral')
ax.invert_yaxis()
ax.set_xlabel('Variable Selection Weight')
ax.set_title('Decoder Variable Importance')
plt.tight_layout()
plt.show()

## 5 — Attention Weight Analysis

In [ ]:
# Temporal attention weights — how much each encoder position is attended to
attention = interpretation['attention'].detach().cpu().numpy()  # (encoder_len + pred_len,)

ENCODER_LEN = 48
PRED_LEN = 24
total_len = ENCODER_LEN + PRED_LEN

fig, ax = plt.subplots(figsize=(14, 4))
x = np.arange(len(attention))
colors = ['steelblue'] * ENCODER_LEN + ['coral'] * PRED_LEN
ax.bar(x, attention, color=colors[:len(attention)], alpha=0.8)
ax.axvline(ENCODER_LEN - 0.5, color='black', ls='--', lw=1, label='Encoder / Decoder boundary')
ax.set_xlabel('Time Position (encoder → decoder)')
ax.set_ylabel('Mean Attention Weight')
ax.set_title('Temporal Self-Attention Pattern')
ax.legend()
plt.tight_layout()
plt.show()

# Highlight which encoder positions get most attention
enc_attn = attention[:ENCODER_LEN]
top_pos = np.argsort(enc_attn)[::-1][:5]
print('Top encoder positions (0=oldest, 47=most recent):')
for pos in top_pos:
    lag = ENCODER_LEN - pos
    print(f'  Position {pos} (t-{lag}h): weight={enc_attn[pos]:.4f}')

## 6 — Discovered Bias vs Hardcoded BIAS_TABLE

In [ ]:
# The hardcoded BIAS_TABLE from bias_correction.py
from bias_correction import BIAS_TABLE  # type: ignore[import-not-found]

# Reconstruct as DataFrame
hardcoded_bias = pd.DataFrame(
    {dow: {h: BIAS_TABLE.get((h, dow), 0) for h in range(24)} for dow in range(7)}
)
hardcoded_bias.columns = dow_labels
hardcoded_bias.index.name = 'hour'

# Discovered bias (from TFT predictions on test set)
discovered_bias = tft_bias_pivot.copy()

# Difference
diff = discovered_bias - hardcoded_bias

fig, axes = plt.subplots(1, 3, figsize=(20, 9))

for ax, data, title in zip(axes, 
    [hardcoded_bias, discovered_bias, diff],
    ['Hardcoded BIAS_TABLE (MW)', 'Discovered TFT Bias (MW)', 'Difference (Discovered − Hardcoded)']
):
    vmax = max(abs(data.values.min()), abs(data.values.max()))
    sns.heatmap(data, annot=True, fmt='.0f', cmap='RdBu_r',
                center=0, vmin=-vmax, vmax=vmax, ax=ax)
    ax.set_title(title, fontsize=11)
    ax.set_ylabel('Hour')

plt.suptitle('Bias Comparison: Hardcoded vs Discovered', fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

print(f'Hardcoded table mean absolute bias:  {hardcoded_bias.abs().values.mean():.0f} MW')
print(f'Discovered TFT mean absolute bias:   {discovered_bias.abs().values.mean():.0f} MW')
print(f'Mean absolute difference:            {diff.abs().values.mean():.0f} MW')

In [ ]:
# Scatter: hardcoded vs discovered bias
hc_flat = hardcoded_bias.values.flatten()
disc_flat = discovered_bias.values.flatten()

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(hc_flat, disc_flat, alpha=0.5, s=20)
lims = [min(hc_flat.min(), disc_flat.min()) - 100, max(hc_flat.max(), disc_flat.max()) + 100]
ax.plot(lims, lims, 'k--', alpha=0.3, label='y=x')
ax.set_xlabel('Hardcoded BIAS_TABLE (MW)')
ax.set_ylabel('Discovered TFT Bias (MW)')
ax.set_title('Hardcoded vs Discovered Bias (168 cells)')
ax.legend()

corr = np.corrcoef(hc_flat, disc_flat)[0, 1]
ax.text(0.05, 0.95, f'r = {corr:.3f}', transform=ax.transAxes, fontsize=11,
        verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
plt.tight_layout()
plt.show()

## 7 — Worst Predictions Deep-Dive

In [ ]:
# Top 20 worst absolute errors
worst = step_agg.nlargest(20, 'abs_error')[['timestamp', 'actual', 'pred', 'error', 'abs_error', 'hour', 'dow']].copy()
worst['dow_name'] = worst['dow'].map(dict(enumerate(dow_labels)))
worst = worst.reset_index(drop=True)
worst.index += 1
worst

In [ ]:
# Visualise a sample bad day
worst_day = worst.iloc[0]['timestamp'].normalize()
day_mask = (step_agg['timestamp'] >= worst_day) & (step_agg['timestamp'] < worst_day + pd.Timedelta(days=1))
day_data = step_agg[day_mask].sort_values('timestamp')

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(day_data['timestamp'], day_data['actual'], 'k-o', ms=4, label='Actual')
ax.plot(day_data['timestamp'], day_data['pred'], 'r--s', ms=4, label='TFT Prediction')
ax.fill_between(day_data['timestamp'], day_data['actual'], day_data['pred'], alpha=0.2, color='red')
ax.set_ylabel('Demand (MW)')
ax.set_title(f'Worst Day: {worst_day.date()}')
ax.legend()
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Error distribution (overall)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram
axes[0].hist(step_agg['error'], bins=80, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].axvline(0, color='black', ls='-', lw=0.8)
axes[0].axvline(step_agg['error'].mean(), color='red', ls='--', label=f'mean={step_agg["error"].mean():.0f}')
axes[0].set_xlabel('Error (MW)')
axes[0].set_ylabel('Count')
axes[0].set_title('Error Distribution')
axes[0].legend()

# QQ plot
from scipy import stats
stats.probplot(step_agg['error'], dist='norm', plot=axes[1])
axes[1].set_title('Q–Q Plot')

plt.tight_layout()
plt.show()

print(f'Error stats: mean={step_agg["error"].mean():.1f}, std={step_agg["error"].std():.1f}, '
      f'skew={step_agg["error"].skew():.2f}, kurtosis={step_agg["error"].kurtosis():.2f}')

## Summary

**Key findings to revisit after training:**

1. **Baseline floor**: Persistence-168h gives ~4.7% MAPE — TFT should beat this significantly
2. **Hour × DoW bias**: The bias heatmap shows whether the model learns the weekday/weekend transition
   or whether the hardcoded BIAS_TABLE is still needed
3. **Feature importance**: Encoder variable selection reveals which features the model actually uses
4. **Attention**: Where the model looks in the encoder window — expect peaks at t-24h and t-168h
5. **Worst predictions**: Usually around holidays or extreme weather — these edge cases
   inform whether we need holiday-specific or temperature-interaction features